# GENE CRISpy — M1 Genetics Foundations

Independent study aid mapped to selected public UIC College of Medicine Block 1 objectives. Not UIC-authored or endorsed; not a complete syllabus. Synthetic examples only. No clinical interpretation, patient risk scoring, variant annotation or CRISPR design.

Source: https://medicine.uic.edu/education/md/md-curriculum/phase-1-curriculum/

Reviewed 2026-11-14. Phase 1 includes M1 and M2; this notebook covers selected Block 1 connections only.


## 1. Load the core package
CPU is sufficient. Clone the reviewed main commit and apply the tested learning update embedded in this notebook. No Firebase, API server, credentials or GPU is required.


In [6]:
from pathlib import Path
import subprocess
import sys
import tempfile

# Create a new directory
repo = Path(tempfile.mkdtemp(
    prefix="genecrispy-m1-", dir="/content"
)) / "repository"
repo.mkdir(parents=True, exist_ok=True)

# Initialize a repository
subprocess.run(["git", "init", str(repo)], check=True)

# Add the remote
subprocess.run([
    "git", "-C", str(repo), "remote", "add", "origin",
    "https://github.com/contrivancecompanychicago/replicatedgenecrispy2dpatientworld.git"
], check=True)

# Enable sparse-checkout
subprocess.run(["git", "-C", str(repo), "config", "core.sparseCheckout", "true"], check=True)

# Specify directories to pull (avoiding the problematic examples folder)
sparse_file = repo / ".git/info/sparse-checkout"
with sparse_file.open("w", encoding="utf-8") as f:
    f.write("genecrispy/\ntests/\n")

# Fetch the target branch
subprocess.run([
    "git", "-C", str(repo), "fetch", "--depth=1", "origin", "genecrispy-m1-genetics"
], check=True)

# Checkout the fetched branch
subprocess.run([
    "git", "-C", str(repo), "checkout", "genecrispy-m1-genetics"
], check=True)

# Load the updated package instead of any earlier imported copy.
for name in list(sys.modules):
    if name == "genecrispy" or name.startswith("genecrispy."):
        del sys.modules[name]

sys.path.insert(0, str(repo))

from genecrispy import GeneCrispyAgent, __version__
from genecrispy import (
    get_m1_genetics_curriculum,
    get_genetics_lesson,
)

agent = GeneCrispyAgent()

print("GENE CRISpy version:", __version__)
print("M1 genetics lessons:",
      len(get_m1_genetics_curriculum()["lessons"]))

GENE CRISpy version: 0.2.0
M1 genetics lessons: 5


## 2. Read the learning map
The objective connections below are our study mapping, not additional official UIC objectives. B1.4 connects inheritance and genetic testing; B1.8 includes pharmacogenomics; B1.11 supports evidence appraisal; B1.13 includes ethical and structural context.


In [7]:
curriculum = get_m1_genetics_curriculum()
print(curriculum["notice"])
print("Source:", curriculum["source_url"])
for lesson in curriculum["lessons"]:
    print(lesson["id"], "—", lesson["title"], "—", ", ".join(lesson["objective_ids"]))


Independent GENE CRISpy study aid; not affiliated with or endorsed by UIC. Public-objective mapping only, not a complete syllabus or clinical interpretation.
Source: https://medicine.uic.edu/education/md/md-curriculum/phase-1-curriculum/
sequence_foundations — DNA, RNA and base composition — B1.4
inheritance — Inheritance and family history — B1.4
testing_limits — Genetic testing and evidence limits — B1.4, B1.11
pharmacogenomics — Pharmacogenomics as a learning connection — B1.8
ethics — Genetics, consent and equitable access — B1.13


## 3. Repeat your three DNA examples and add RNA
FASTA is a text format. GC percentage is a composition statistic, not a measure of disease risk. GCGCGC does not establish DNA versus RNA identity from letters alone.


In [8]:
samples = ">sample_A\nACGTGC\n>sample_B\nATATAT\n>sample_C\nGCGCGC\n>sample_RNA\nACGU\n"
results = agent.analyze(samples, input_format="fasta")
for record in results["records"]:
    print(record["id"], record["molecule_type"], "length:", record["length"],
          "GC%:", record["gc_percent_of_canonical_bases"])
assert results["records"][3]["molecule_type"] == "RNA"


sample_A DNA length: 6 GC%: 66.67
sample_B DNA length: 6 GC%: 0.0
sample_C nucleic_acid_unspecified length: 6 GC%: 100.0
sample_RNA RNA length: 4 GC%: 50.0


## 4. Learn and reflect
Read each explanation, write your own response, then compare with the model answer in the next cell. No learner responses are collected or sent anywhere.


In [9]:
for lesson in curriculum["lessons"]:
    print("\n" + lesson["title"])
    print(lesson["explanation"])
    print("Reflect:", lesson["reflection"])



DNA, RNA and base composition
DNA uses A, C, G and T; RNA uses A, C, G and U. GC percentage describes the fraction of canonical bases that are G or C. A short sequence containing neither T nor U may not distinguish DNA from RNA. Sequence alphabets can also overlap with protein alphabets.
Reflect: Why can GCGCGC have 100% GC without establishing whether it is DNA or RNA?

Inheritance and family history
In a simple autosomal recessive Aa by Aa model, each parent contributes A or a with equal probability. Offspring genotype probabilities are AA: 25%, Aa: 50%, aa: 25%. This assumes Mendelian segregation; phenotype also depends on the model's penetrance and other assumptions. This is not a calculation for an actual family.
Reflect: Does one child's genotype change the probabilities for the next independent conception?

Genetic testing and evidence limits
A FASTA summary does not identify a gene or a pathogenic variant. Interpretation needs context such as a reference sequence, genomic loca

In [10]:
for lesson in curriculum["lessons"]:
    print(lesson["id"] + ":", lesson["answer"])


sequence_foundations: G and C occur in both DNA and RNA; molecule identity requires additional context.
inheritance: No. Under these assumptions the genotype probabilities are the same for each conception.
testing_limits: No. Matching composition does not establish sequence identity, function or clinical significance.
pharmacogenomics: Validated variant identification, relevant evidence and clinical context, among other requirements.
ethics: Genetic data can reveal sensitive information about individuals and biological relatives.


## 5. Check the genetics learning path
These focused checks do not load the legacy PatientWorld/API modules. The pre-existing full test suite still has unrelated import blockers.


In [11]:
subprocess.run([
    sys.executable, "-m", "unittest", "discover",
    "-s", "tests", "-p", "test_m1_education.py", "-v"
], cwd=repo, check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_m1_education.py', '-v'], returncode=0)

## 6. Download your results (optional)
Downloads a CSV and JSON bundle. Save a copy of the notebook to your Google Drive as well; runtime files are temporary.


In [ ]:
import csv
import json
import zipfile
from google.colab import files

csv_path = Path("/content/GENE_CRISpy_M1_results.csv")
fields = ["id", "molecule_type", "length", "gc_percent_of_canonical_bases", "ambiguous_symbol_count"]
with csv_path.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=fields, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(results["records"])
json_path = Path("/content/GENE_CRISpy_M1_learning.json")
json_path.write_text(json.dumps({"curriculum": curriculum, "results": results}, indent=2))
bundle = Path("/content/GENE_CRISpy_M1_results.zip")
with zipfile.ZipFile(bundle, "w") as archive:
    archive.write(csv_path, csv_path.name)
    archive.write(json_path, json_path.name)
files.download(str(bundle))


In [12]:
%pip -q install google-cloud-firestore

from google.colab import auth
from google.cloud import firestore

auth.authenticate_user()
db = firestore.Client(project="uicpbvls", database="(default)")

# Check database access without writing any data.
list(db.collection("genecrispy_runs").limit(1).stream())

print("Connected to Firestore: uicpbvls")

Connected to Firestore: uicpbvls
